# OCR & Document Understanding Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: CTC loss + greedy decoder

In [ ]:
```python

import torch

import torch.nn as nn

import torch.nn.functional as F

def ctc_loss(log_probs, targets, input_lengths, target_lengths, blank=0):

    """

    log_probs:      (T, N, C) log-softmax over vocab including blank at index 0

    targets:        (N, S) int targets (no blanks)

    input_lengths:  (N,) per-sample time steps used

    target_lengths: (N,) per-sample target length

    """

    return F.ctc_loss(log_probs, targets, input_lengths, target_lengths,

                      blank=blank, reduction="mean", zero_infinity=True)

def greedy_ctc_decode(log_probs, blank=0):

    """

    log_probs: (T, N, C) log-softmax

    returns: list of index sequences (blanks removed, repeats merged)

    """

    preds = log_probs.argmax(dim=-1).transpose(0, 1).cpu().tolist()

    out = []

    for seq in preds:

        decoded = []

        prev = None

        for idx in seq:

            if idx != prev and idx != blank:

                decoded.append(idx)

            prev = idx

        out.append(decoded)

    return out

In [ ]:
```

`F.ctc_loss` uses the efficient CuDNN implementation when available. The greedy decoder is simpler than a beam search and usually within 1% CER of it.

### Step 2: Tiny CRNN recogniser

Minimal CNN + BiLSTM for line OCR.

In [ ]:
```python

class TinyCRNN(nn.Module):

    def __init__(self, vocab_size=40, hidden=128, feat=32):

        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(1, feat, 3, 1, 1), nn.BatchNorm2d(feat), nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            nn.Conv2d(feat, feat * 2, 3, 1, 1), nn.BatchNorm2d(feat * 2), nn.ReLU(inplace=True),

            nn.MaxPool2d(2),

            nn.Conv2d(feat * 2, feat * 4, 3, 1, 1), nn.BatchNorm2d(feat * 4), nn.ReLU(inplace=True),

            nn.MaxPool2d((2, 1)),

            nn.Conv2d(feat * 4, feat * 4, 3, 1, 1), nn.BatchNorm2d(feat * 4), nn.ReLU(inplace=True),

            nn.MaxPool2d((2, 1)),

        )

        self.rnn = nn.LSTM(feat * 4, hidden, bidirectional=True, batch_first=True)

        self.head = nn.Linear(hidden * 2, vocab_size)

    def forward(self, x):

        # x: (N, 1, H, W)

        f = self.cnn(x)                # (N, C, H', W')

        f = f.mean(dim=2).transpose(1, 2)  # (N, W', C)

        h, _ = self.rnn(f)

        return F.log_softmax(self.head(h).transpose(0, 1), dim=-1)  # (W', N, vocab)

In [ ]:
```

Fixed-height input (the CNN max-pools height to 1). Width is the time dimension for CTC.

### Step 3: Synthetic OCR

Generate black-on-white digit strings for an end-to-end smoke test.

In [ ]:
```python

import numpy as np

def synthetic_line(text, height=32, char_width=16):

    W = char_width * len(text)

    img = np.ones((height, W), dtype=np.float32)

    for i, c in enumerate(text):

        x = i * char_width

        shade = 0.0 if c.isalnum() else 0.5

        img[6:height - 6, x + 2:x + char_width - 2] = shade

    return img

def build_batch(strings, vocab):

    H = 32

    W = 16 * max(len(s) for s in strings)

    imgs = np.ones((len(strings), 1, H, W), dtype=np.float32)

    target_lengths = []

    targets = []

    for i, s in enumerate(strings):

        imgs[i, 0, :, :16 * len(s)] = synthetic_line(s)

        ids = [vocab.index(c) for c in s]

        targets.extend(ids)

        target_lengths.append(len(ids))

    return torch.from_numpy(imgs), torch.tensor(targets), torch.tensor(target_lengths)

vocab = ["_"] + list("0123456789abcdefghijklmnopqrstuvwxyz")

imgs, targets, lengths = build_batch(["hello", "world"], vocab)

print(f"images: {imgs.shape}   targets: {targets.shape}   lengths: {lengths.tolist()}")

In [ ]:
```

A real OCR dataset adds fonts, noise, rotation, blur, and colour. The pipeline above is identical.

### Step 4: Training sketch

In [ ]:
```python

model = TinyCRNN(vocab_size=len(vocab))

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(200):

    strings = ["abc" + str(step % 10)] * 4 + ["xyz" + str((step + 1) % 10)] * 4

    imgs, targets, target_lens = build_batch(strings, vocab)

    log_probs = model(imgs)  # (W', 8, vocab)

    input_lens = torch.full((8,), log_probs.size(0), dtype=torch.long)

    loss = ctc_loss(log_probs, targets, input_lens, target_lens, blank=0)

    opt.zero_grad(); loss.backward(); opt.step()

In [ ]:
```

Loss should drop from ~3 to ~0.2 over 200 steps on this trivial synthetic data.

## Exercises

In [ ]:
1. **(Easy)** Train the TinyCRNN on 5-digit random numeric strings for 500 steps. Report CER on a held-out set.
2. **(Medium)** Replace greedy decoding with beam search (beam_width=5). Report CER delta. On which inputs does beam search win?
3. **(Hard)** Use PaddleOCR on a set of 20 receipts, extract line items, and compute F1 against hand-labelled ground truth for {item_name, price} pairs.